In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import sys, pickle, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

sys.path.append('../datasets/spe-1/spe1_helper_modules/')
from config import (SPE1_PICKLE_ROOT, DICT_CELL_TYPE, DICT_PATCH_TYPE,
                    DICT_CORT_DEPTH, DICT_DARK_NEURONS, DICT_EAP_WAV)

# Cell-type colours matching Figure 4C of Marques-Smith et al. 2020
COL_PC  = '#CC44CC'   # magenta  — pyramidal cells
COL_IN  = '#00CCCC'   # cyan     — interneurons

COL_EAP = '#009E73'   # green    — clear EAP
COL_NO  = '#888888'   # grey     — no EAP / recording issue

FS_TITLE  = 11
FS_LABEL  = 10
FS_TICK   = 9
FS_ANNOT  = 8
import os


In [ ]:
# ── Run-control flags ─────────────────────────────────────────────────────────
FORCE_RERUN = False   # set True to recompute and overwrite pickles


In [ ]:
# ── Build metadata DataFrame ───────────────────────────────────────────────────
rows = []
for pkl in sorted(glob.glob(SPE1_PICKLE_ROOT + '/cluster_pickles/c*_cluster_df.pkl')):
    cnum = int(pkl.split('/')[-1].replace('_cluster_df.pkl','').lstrip('c'))
    df   = pickle.load(open(pkl, 'rb'))
    t    = df['spk_times_ms'].values
    rows.append(dict(
        cell_num     = cnum,
        cell_id      = f'c{cnum}',
        cell_type    = DICT_CELL_TYPE.get(cnum, 'unknown'),
        patch_type   = DICT_PATCH_TYPE.get(cnum, 'unknown'),
        cort_depth   = DICT_CORT_DEPTH.get(cnum, np.nan),
        dark_neuron  = DICT_DARK_NEURONS.get(cnum, False),
        eap_visible  = DICT_EAP_WAV.get(cnum, False),
        n_spikes     = len(df),
        duration_min = (t.max() - t.min()) / 1000 / 60,
        firing_rate  = len(df) / ((t.max() - t.min()) / 1000),
    ))

meta = pd.DataFrame(rows)
meta['method'] = meta['patch_type'].apply(
    lambda x: 'Whole-cell' if 'WC' in str(x) else 'Juxtacellular'
)

# Three-way EAP status per paper definition:
# - Clear EAP: PSTA shows detectable canonical waveform
# - Dark neuron candidate: close to probe, good session, >200 spikes — no EAP despite ideal conditions
# - No EAP / recording issue: lack of EAP likely explained by probe drift or misalignment
def _eap_status(row):
    if row['eap_visible']:
        return 'Clear EAP'
    elif row['dark_neuron']:
        return 'Dark neuron\ncandidate'
    else:
        return 'No EAP\n(recording issue)'
meta['eap_status'] = meta.apply(_eap_status, axis=1)

print(f'n = {len(meta)} cells')
print(meta['eap_status'].value_counts().to_string())

In [ ]:
# ── Figure: dataset summary ────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 8))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.38)

# ── A. Cell type breakdown ─────────────────────────────────────────────────────
ax_a = fig.add_subplot(gs[0, 0])
ct_counts = meta['cell_type'].value_counts()
bars = ax_a.bar(ct_counts.index, ct_counts.values,
                color=[COL_PC if c == 'PC' else COL_IN for c in ct_counts.index],
                edgecolor='white', width=0.55)
for bar, val in zip(bars, ct_counts.values):
    ax_a.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
              str(val), ha='center', va='bottom', fontsize=FS_ANNOT, fontweight='bold')
ax_a.set_ylabel('N cells', fontsize=FS_LABEL)
ax_a.set_title('A.  Cell type', fontsize=FS_TITLE, fontweight='bold', loc='left')
ax_a.set_ylim(0, ct_counts.max() * 1.2)
sns.despine(ax=ax_a)

# ── B. Recording method ────────────────────────────────────────────────────────
ax_b = fig.add_subplot(gs[0, 1])
method_ct = meta.groupby(['method','cell_type']).size().unstack(fill_value=0)
bot = np.zeros(len(method_ct))
for ct, col in [('PC', COL_PC), ('IN', COL_IN)]:
    if ct in method_ct.columns:
        vals = method_ct[ct].values
        ax_b.bar(method_ct.index, vals, bottom=bot, color=col,
                 edgecolor='white', width=0.5, label=ct)
        for i, (v, b) in enumerate(zip(vals, bot)):
            if v > 0:
                ax_b.text(i, b + v/2, str(v), ha='center', va='center',
                          fontsize=FS_ANNOT, color='white', fontweight='bold')
        bot += vals
ax_b.set_ylabel('N cells', fontsize=FS_LABEL)
ax_b.set_title('B.  Recording method', fontsize=FS_TITLE, fontweight='bold', loc='left')
ax_b.legend(fontsize=FS_ANNOT, frameon=False, loc='upper right')
ax_b.set_ylim(0, bot.max() * 1.25)
ax_b.tick_params(axis='x', labelsize=FS_TICK)
sns.despine(ax=ax_b)

# ── C. Cortical depth ─────────────────────────────────────────────────────────
ax_c = fig.add_subplot(gs[0, 2])
rng = np.random.default_rng(42)
for ct, col in [('PC', COL_PC), ('IN', COL_IN)]:
    sub = meta[meta.cell_type == ct].dropna(subset=['cort_depth'])
    jitter = rng.uniform(-0.08, 0.08, len(sub))
    ax_c.scatter(jitter + (0 if ct == 'PC' else 1), sub['cort_depth'],
                 color=col, s=40, alpha=0.75, edgecolors='white', lw=0.4, label=ct, zorder=4)
    ax_c.plot([(-0.2 if ct=='PC' else 0.8), (0.2 if ct=='PC' else 1.2)],
              [sub['cort_depth'].median()]*2, color=col, lw=2.5, zorder=5)
ax_c.set_xticks([0, 1])
ax_c.set_xticklabels(['PC', 'IN'], fontsize=FS_TICK)
ax_c.set_ylabel('Cortical depth (µm)', fontsize=FS_LABEL)
ax_c.set_title('C.  Cortical depth', fontsize=FS_TITLE, fontweight='bold', loc='left')
ax_c.invert_yaxis()
sns.despine(ax=ax_c)

# ── D. EAP status (Marques-Smith et al. 2020 classification) ──────────────────
ax_d = fig.add_subplot(gs[0, 3])
STATUS_ORDER = ['Clear EAP', 'Dark neuron\ncandidate', 'No EAP\n(recording issue)']
STATUS_COLS  = {
    'Clear EAP':              COL_EAP,     # green
    'Dark neuron\ncandidate': '#CC79A7',   # pink
    'No EAP\n(recording issue)': COL_NO,  # grey
}
eap_ct = (meta.groupby(['cell_type', 'eap_status']).size()
            .unstack(fill_value=0)
            .reindex(columns=[s for s in STATUS_ORDER if s in meta['eap_status'].unique()],
                     fill_value=0))

x = np.arange(len(eap_ct))
w = 0.22
for i, status in enumerate(eap_ct.columns):
    col  = STATUS_COLS.get(status, '#888')
    vals = eap_ct[status].values
    ax_d.bar(x + i*w, vals, width=w, color=col, edgecolor='white', alpha=0.9, label=status)
    for xi, v in zip(x + i*w, vals):
        if v > 0:
            ax_d.text(xi, v + 0.12, str(v), ha='center', fontsize=FS_ANNOT)

ax_d.set_xticks(x + w*(len(eap_ct.columns)-1)/2)
ax_d.set_xticklabels(eap_ct.index, fontsize=FS_TICK)
ax_d.set_ylabel('N cells', fontsize=FS_LABEL)
ax_d.set_title('D.  EAP on Neuropixels', fontsize=FS_TITLE, fontweight='bold', loc='left')
ax_d.legend(fontsize=FS_ANNOT - 1, frameon=False, loc='upper right')
ax_d.set_ylim(0, eap_ct.values.max() * 1.6)
sns.despine(ax=ax_d)

# ── E. N spikes per cell ──────────────────────────────────────────────────────
ax_e = fig.add_subplot(gs[1, 0:2])
for ct, col in [('PC', COL_PC), ('IN', COL_IN)]:
    sub = meta[meta.cell_type == ct]
    ax_e.hist(sub['n_spikes'], bins=15, color=col, alpha=0.7, edgecolor='white', label=ct)
ax_e.axvline(meta['n_spikes'].median(), color='k', lw=1.5, ls='--', alpha=0.6,
             label=f'Median = {int(meta.n_spikes.median()):,}')
ax_e.set_xlabel('N spikes', fontsize=FS_LABEL)
ax_e.set_ylabel('N cells', fontsize=FS_LABEL)
ax_e.set_title('E.  Spikes per cell', fontsize=FS_TITLE, fontweight='bold', loc='left')
ax_e.legend(fontsize=FS_ANNOT, frameon=False)
ax_e.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
sns.despine(ax=ax_e)

# ── F. Firing rate vs duration ────────────────────────────────────────────────
ax_f = fig.add_subplot(gs[1, 2:4])
for ct, col in [('PC', COL_PC), ('IN', COL_IN)]:
    sub = meta[meta.cell_type == ct]
    ax_f.scatter(sub['duration_min'], sub['firing_rate'],
                 color=col, s=55, alpha=0.8, edgecolors='white', lw=0.5, label=ct, zorder=4)
ax_f.set_xlabel('Recording duration (min)', fontsize=FS_LABEL)
ax_f.set_ylabel('Firing rate (Hz)', fontsize=FS_LABEL)
ax_f.set_title('F.  Recording duration vs firing rate', fontsize=FS_TITLE,
               fontweight='bold', loc='left')
ax_f.legend(fontsize=FS_ANNOT, frameon=False)
sns.despine(ax=ax_f)

fig.suptitle(f'spe-1 dataset  —  n = {len(meta)} cells  '
             f'({(meta.cell_type=="PC").sum()} PC,  {(meta.cell_type=="IN").sum()} IN)',
             fontsize=13, fontweight='bold', y=1.01)

plt.savefig('spe1_dataset_summary.pdf', bbox_inches='tight', dpi=300)
plt.savefig('spe1_dataset_summary.png', bbox_inches='tight', dpi=300)
plt.show()
print('Saved.')

## Panel D — EAP category definitions (Marques-Smith et al. 2020)

EAP detectability was assessed by computing the **patch spike-triggered average (PSTA)** of the Neuropixels signal around every patch-clamp spike. All neurons were within 10–144 µm of the nearest probe channel.

- **Clear EAP** — PSTA reveals a canonical extracellular waveform with peak-peak amplitude > 10 µV. The neuron is detectable by standard spike sorting.

- **Dark neuron candidate** — No clear PSTA waveform despite: (1) being within ~50 µm of the probe, (2) ≥200 spikes recorded, and (3) recorded in a session where other neurons *were* detectable extracellularly. Absence of EAP is unlikely to be a recording artefact — these cells may be genuinely undetectable by extracellular electrodes.

- **No EAP / recording issue** — No clear PSTA waveform, but the absence is likely explained by technical factors: probe drift, manipulator misalignment, or bending of the probe after brain entry (sessions where *no* cells were detectable, or sessions that started with detectable cells and ended without). Not candidates for "dark neuron" classification.